# Setup Instruction

1. Please apply the following cmd to clone all the files from github:

``!git clone -b develop https://github.com/Ach57/pose-form-coach.git``

2. You need to have the following zip files uploaded:
- `` features.zip``  
- `` labels.zip``  
- `` Splits.zip ``
- `` poses.zip ``

# GymForm — OHP Error Detection Training

Train a Causal TCN to detect per-frame elbow and knee errors in overhead press videos.

**Setup:** Upload or mount your `gym-form/` repo (with extracted `data/features/ohp/` and `data/labels/ohp/`).

## 1. Environment Setup

In [ ]:
from pathlib import Path

from src.utils.runtime import ensure_env_setup
from src.utils.hardware import device_verification
from src.utils.dataset_io import prepare_dataset_archives

# ── Environment setup ──
IN_COLAB, REPO_ROOT = ensure_env_setup()

# ── Install dependencies ──
if IN_COLAB:
    !pip install -q pyyaml scipy numpy
    # PyTorch is pre-installed on Colab with GPU support

# ── Device Used ──
DEVICE = device_verification()

# ── Zip files preperation ──
prepare_dataset_archives(Path("/content"))

## 2. Load Configs

In [ ]:
from src.utils.io import load_yaml

dataset_cfg = load_yaml("configs/dataset.ohp.yaml")
model_cfg = load_yaml("configs/model.tcn.yaml")
train_full = load_yaml("configs/train.ohp.yaml")
train_cfg = train_full["train"]
aug_cfg = train_full.get("augmentations", {})
val_cfg = train_full.get("validation", {})

print(f"Exercise:    {dataset_cfg['exercise']}")
print(f"Labels:      {dataset_cfg['labels']}")
print(f"Window:      T={dataset_cfg['window']['T']}, stride={dataset_cfg['window']['stride']}")
print(f"Pos weight:  {train_cfg['pos_weight']}")
print(f"Batch size:  {train_cfg['batch_size']}")
print(f"Epochs:      {train_cfg['epochs']}")

## 3. 3D Skeleton Visualization

Interactive 3D view of MediaPipe pose landmarks for a sample video.
Use the slider to scrub through frames. Bones are colored by body region.

In [ ]:
from src.utils.mediapipe_visualization import build_frames_for_slider

# ── Pick a sample video (one with known errors) ──
SAMPLE_VIDEO = "62805_6"
build_frames_for_slider(dataset_cfg, SAMPLE_VIDEO)

In [ ]:
from src.utils.mediapipe_visualization import build_feature_timeline

build_feature_timeline(dataset_cfg, SAMPLE_VIDEO)

## 4. Build Model

In [ ]:
from src.models.causal_tcn import CausalTCN

model = CausalTCN.from_config("configs/model.tcn.yaml")

n_params = sum(p.numel() for p in model.parameters())
print(f"Model:  CausalTCN")
print(f"Params: {n_params:,}")
print(f"RF:     {model.receptive_field} frames ({model.receptive_field/30:.2f}s @ 30fps)")
print(f"Input:  (B, T, {model.in_features})")
print(f"Output: (B, T, {model.n_labels})")

## 5. Train

In [ ]:
from src.train.trainer import Trainer

# ── Overrides (adjust for Colab resources) ──
# train_cfg["batch_size"] = 128   # bump if GPU memory allows
# train_cfg["epochs"] = 80        # train longer if needed

trainer = Trainer(
    model=model,
    train_cfg=train_cfg,
    dataset_cfg=dataset_cfg,
    aug_cfg=aug_cfg,
    val_cfg=val_cfg,
    device=DEVICE,
    checkpoint_dir="checkpoints",
)

print(f"Loss:      BCEWithLogitsLoss (pos_weight={train_cfg['pos_weight']})")
print(f"Optimizer: AdamW (lr={train_cfg['lr']})")
print(f"Patience:  {trainer.patience} epochs")
print(f"Device:    {DEVICE}")

In [ ]:
# ── Run training ──
history = trainer.fit()

## 6. Training Curves

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Loss
axes[0].plot(history["train_loss"], label="Train")
axes[0].plot(history["val_loss"], label="Val")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("BCE Loss")
axes[0].set_title("Loss")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Segment-mAP
axes[1].plot(history["segment_map"], label="Val seg-mAP", color="green")
best_ep = max(range(len(history["segment_map"])), key=lambda i: history["segment_map"][i])
axes[1].axvline(best_ep, color="red", linestyle="--", alpha=0.5, label=f"Best (epoch {best_ep+1})")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Segment mAP")
axes[1].set_title("Validation Segment-mAP")
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\nBest segment-mAP: {max(history['segment_map']):.4f} at epoch {best_ep+1}")

## 7. Evaluate Best Checkpoint

In [ ]:
# Load best checkpoint
trainer.load_checkpoint("best.pt")

# Run on test split
from pathlib import Path
from src.datasets.window_dataset import WindowDataset
from src.train.trainer import _make_collate
from torch.utils.data import DataLoader
import numpy as np

test_ds = WindowDataset.from_config(
    dataset_cfg,
    split="test",
    features_dir=Path(dataset_cfg["paths"]["features_dir"]),
    labels_dir=Path(dataset_cfg["paths"]["labels_dir"]),
)
test_loader = DataLoader(
    test_ds, batch_size=train_cfg["batch_size"],
    shuffle=False, collate_fn=_make_collate(),
)
print(f"Test windows: {len(test_ds)}")

In [ ]:
from src.eval.metrics import frame_metrics, hysteresis_segments, segment_map
from src.train.trainer import _binary_to_segments

model.eval()
all_probs, all_labels = [], []

with torch.no_grad():
    for feats, labels in test_loader:
        feats = feats.to(DEVICE)
        logits = model(feats)
        probs = torch.sigmoid(logits).cpu().numpy()
        all_probs.append(probs)
        all_labels.append(labels.numpy())

probs_cat = np.concatenate(all_probs)   # (N, T, 2)
labels_cat = np.concatenate(all_labels)

# Frame metrics
fm = frame_metrics(probs_cat, labels_cat)
print(f"Frame-level — P: {fm['precision']:.3f}  R: {fm['recall']:.3f}  F1: {fm['f1']:.3f}")

# Segment-mAP per label
label_names = dataset_cfg["labels"]
tiou = val_cfg.get("tiou", [0.1, 0.25, 0.5])

for li, name in enumerate(label_names):
    pred_segs_all, scores_all, gt_segs_all = [], [], []
    for i in range(probs_cat.shape[0]):
        p = probs_cat[i, :, li]
        g = labels_cat[i, :, li]
        segs = hysteresis_segments(p, on=0.5, off=0.3, min_dur=3)
        scores = [float(p[s:e+1].mean()) for s, e in segs]
        pred_segs_all.append(segs)
        scores_all.append(scores)
        gt_segs_all.append(_binary_to_segments(g))
    smap = segment_map(pred_segs_all, scores_all, gt_segs_all, tiou)
    print(f"{name}: {smap}")

## 8. Save to Drive

In [ ]:
from pathlib import Path
from src.utils.checkpoints import save_checkpoints

save_checkpoints(
    checkpoints_dir=Path("checkpoints"),
    in_colab=IN_COLAB,
    drive_dest=Path("/content/drive/MyDrive/gym-form-checkpoints")
)
